In [ ]:
!pip install mtcnn -q
import cv2
import os
import time

from mtcnn import MTCNN
from google.colab import files
from google.colab.patches import cv2_imshow

In [ ]:
print("Please upload an image...")

uploaded = files.upload()

if len(uploaded) == 0:
    print("No image was uploaded.")
else:
    image_path = next(iter(uploaded))
    print(f"Successfully uploaded: {image_path}")


In [ ]:
image = cv2.imread(image_path)

if image is None:
    print("Error: Could not read the image.")
else:
    height, width, channels = image.shape

    print("Image loaded successfully!")
    print(f"Width      : {width}px")
    print(f"Height     : {height}px")
    print(f"Channels   : {channels}")

    cv2_imshow(image)
def preprocess_image(image, max_width=1600):

    original_height, original_width = image.shape[:2]

    # Resize very large images
    if original_width > max_width:

        scale = max_width / original_width

        new_width = int(original_width * scale)
        new_height = int(original_height * scale)

        image = cv2.resize(
            image,
            (new_width, new_height),
            interpolation=cv2.INTER_AREA
        )

    # Mild sharpening
    blurred = cv2.GaussianBlur(
        image,
        (0, 0),
        1.0
    )

    enhanced = cv2.addWeighted(
        image,
        1.3,
        blurred,
        -0.3,
        0
    )

    return enhanced


In [ ]:
detector = MTCNN()

print("MTCNN face detector initialized successfully.")
def preprocess_image(image, max_width=1600):

    original_height, original_width = image.shape[:2]

    # Resize large images
    if original_width > max_width:

        scale = max_width / original_width

        new_width = int(original_width * scale)
        new_height = int(original_height * scale)

        image = cv2.resize(
            image,
            (new_width, new_height),
            interpolation=cv2.INTER_AREA
        )

    # Mild sharpening
    blurred = cv2.GaussianBlur(
        image,
        (0, 0),
        1.0
    )

    enhanced = cv2.addWeighted(
        image,
        1.3,
        blurred,
        -0.3,
        0
    )

    return enhanced


# Create processed image
processed_image = preprocess_image(image)

print("Image preprocessing completed.")

cv2_imshow(processed_image)


In [ ]:
rgb_image = cv2.cvtColor(
    processed_image,
    cv2.COLOR_BGR2RGB
)

print("BGR → RGB conversion completed.")
start_time = time.time()

detections = detector.detect_faces(rgb_image)

detection_time = time.time() - start_time

print(f"Detection completed in {detection_time:.3f} seconds")
print(f"Raw faces detected: {len(detections)}")
CONFIDENCE_THRESHOLD = 0.90
MIN_FACE_SIZE = 30

valid_faces = []

height, width = processed_image.shape[:2]

for face in detections:

    confidence = float(face["confidence"])

    x, y, w, h = face["box"]

    # Fix negative coordinates
    x = max(0, x)
    y = max(0, y)

    # Keep box inside image
    x2 = min(width, x + w)
    y2 = min(height, y + h)

    w = x2 - x
    h = y2 - y

    # Confidence filter
    if confidence < CONFIDENCE_THRESHOLD:
        continue

    # Small face filter
    if w < MIN_FACE_SIZE or h < MIN_FACE_SIZE:
        continue

    # Invalid box
    if w <= 0 or h <= 0:
        continue

    valid_faces.append({
        "box": (x, y, w, h),
        "confidence": confidence,
        "landmarks": face.get("keypoints", {})
    })

print(f"Valid faces: {len(valid_faces)}")


In [ ]:
result_image = processed_image.copy()

for i, face in enumerate(valid_faces):

    x, y, w, h = face["box"]
    confidence = face["confidence"]

    x2 = x + w
    y2 = y + h

    # Draw bounding box
    cv2.rectangle(
        result_image,
        (x, y),
        (x2, y2),
        (0, 255, 0),
        3
    )

    # Face label
    label = f"Face {i + 1}: {confidence * 100:.1f}%"

    cv2.putText(
        result_image,
        label,
        (x, max(25, y - 10)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        (0, 255, 0),
        2
    )

print("Bounding boxes added.")

cv2_imshow(result_image)
landmark_image = result_image.copy()

for face in valid_faces:

    landmarks = face["landmarks"]

    for name, point in landmarks.items():

        px, py = point

        cv2.circle(
            landmark_image,
            (px, py),
            5,
            (0, 0, 255),
            -1
        )

        cv2.putText(
            landmark_image,
            name,
            (px + 5, py - 5),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.35,
            (255, 0, 0),
            1
        )

cv2_imshow(landmark_image)
crop_folder = "detected_faces"

os.makedirs(crop_folder, exist_ok=True)

for i, face in enumerate(valid_faces):

    x, y, w, h = face["box"]

    face_crop = processed_image[
        y:y+h,
        x:x+w
    ]

    crop_path = os.path.join(
        crop_folder,
        f"face_{i + 1}.jpg"
    )

    cv2.imwrite(
        crop_path,
        face_crop,
        [cv2.IMWRITE_JPEG_QUALITY, 95]
    )

    print(f"Saved: {crop_path}")
output_path = "output_faces_mtcnn.jpg"

cv2.imwrite(
    output_path,
    result_image,
    [cv2.IMWRITE_JPEG_QUALITY, 95]
)

print(f"Result saved as: {output_path}")
print("\n" + "=" * 55)
print("           FACE DETECTION REPORT")
print("=" * 55)

print(f"Input image          : {image_path}")
print(f"Image width          : {width}px")
print(f"Image height         : {height}px")
print(f"Raw detections       : {len(detections)}")
print(f"Valid faces          : {len(valid_faces)}")
print(f"Confidence threshold : {CONFIDENCE_THRESHOLD * 100:.0f}%")
print(f"Minimum face size    : {MIN_FACE_SIZE}px")
print(f"Detection time       : {detection_time:.3f} seconds")

if len(valid_faces) > 0:

    average_confidence = sum(
        face["confidence"]
        for face in valid_faces
    ) / len(valid_faces)

    print(
        f"Average confidence  : "
        f"{average_confidence * 100:.2f}%"
    )

print(f"Output file          : {output_path}")

print("=" * 55)
